# 01 — Audit des données brutes

**Objectif :** comprendre les 6 fichiers sources et identifier *tous* les problèmes de qualité **avant** de nettoyer quoi que ce soit. Chaque décision de nettoyage appliquée dans `src/etl.py` est justifiée ici.

**Données :** dataset Kaggle *Global Fashion Retail Sales* (synthétique) — voir `data/README.md`.

| Section | Question |
|---|---|
| 1 | Que contient chaque table ? |
| 2 | Où sont les valeurs manquantes, et peut-on les récupérer ? |
| 3 | Y a-t-il des doublons ? |
| 4 | Les valeurs sont-elles dans des plages plausibles ? |
| 5 | Les montants sont-ils cohérents (`Line Total = Prix x Quantité x (1 - Remise)`) ? |
| 6 | Les clés étrangères sont-elles toutes valides ? |
| 7 | Synthèse : problèmes -> décisions |

> Les tables `customers` et `employees` contiennent des noms, emails et téléphones (fictifs). Par principe, ces colonnes ne sont jamais affichées dans ce notebook.

In [1]:
import sys
sys.path.append("..")  # rend le package src/ importable depuis notebooks/

import pandas as pd

from src import etl

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)

In [2]:
raw = etl.load_raw_data()
customers, products, discounts, employees, stores, transactions = (
    raw[name] for name in etl.RAW_TABLES
)

## 1. Vue d'ensemble des tables

In [3]:
pd.DataFrame({
    name: {
        "Lignes": len(df),
        "Colonnes": df.shape[1],
        "Mémoire (Mo)": round(df.memory_usage(deep=True).sum() / 1024**2, 1),
    }
    for name, df in raw.items()
}).T.astype({"Lignes": int, "Colonnes": int})

,Lignes,Colonnes,Mémoire (Mo)
customers,1643306,9,254.50
products,17940,12,6.00
discounts,181,6,0.00
employees,404,4,0.00
stores,35,8,0.00
transactions,6416827,19,"1,262.50"


In [4]:
transactions.head()

,Invoice ID,Line,Customer ID,Product ID,Size,Color,Unit Price,Quantity,Date,Discount,Line Total,Store ID,Employee ID,Currency,Currency Symbol,SKU,Transaction Type,Payment Method,Invoice Total
0,INV-US-001-03558761,1,47162,485,M,NaN,80.50,1,2023-01-01 15:42:00,0.00,80.50,1,7,USD,$,MASU485-M-,Sale,Cash,126.70
1,INV-US-001-03558761,2,47162,2779,G,NaN,31.50,1,2023-01-01 15:42:00,0.40,18.90,1,7,USD,$,CHCO2779-G-,Sale,Cash,126.70
2,INV-US-001-03558761,3,47162,64,M,NEUTRAL,45.50,1,2023-01-01 15:42:00,0.40,27.30,1,7,USD,$,MACO64-M-NEUTRAL,Sale,Cash,126.70
3,INV-US-001-03558762,1,10142,131,M,BLUE,70.00,1,2023-01-01 20:04:00,0.40,42.00,1,6,USD,$,FECO131-M-BLUE,Sale,Cash,77.00
4,INV-US-001-03558762,2,10142,716,L,WHITE,26.00,1,2023-01-01 20:04:00,0.00,26.00,1,6,USD,$,MAT-716-L-WHITE,Sale,Cash,77.00


In [5]:
transactions["Date"].agg(["min", "max"])

min   2023-01-01 00:00:00
max   2025-03-18 20:59:00
Name: Date, dtype: datetime64[us]

- **Grain de `transactions` :** une ligne de facture (`Invoice ID` + `Line`). Les factures de vente commencent par `INV-`, les retours par `RET-`.
- **Période couverte :** du 1er janvier 2023 au 18 mars 2025, soit ~26,5 mois. **Mars 2025 est incomplet** : il devra être exclu de toute analyse mensuelle.
- Les ventes sont exprimées dans **4 devises** (`Currency`) : une conversion en USD sera indispensable avant toute agrégation.

In [6]:
transactions["Currency"].value_counts(normalize=True).rename("Part des lignes")

Currency
EUR   0.40
USD   0.26
CNY   0.24
GBP   0.10
Name: Part des lignes, dtype: float64

## 2. Valeurs manquantes

In [7]:
missing = pd.concat(
    {
        name: pd.DataFrame({"Manquants": df.isna().sum(), "%": df.isna().mean() * 100})
        for name, df in raw.items()
    },
    names=["Table", "Colonne"],
)
missing[missing["Manquants"] > 0].sort_values("%", ascending=False)

Manquants     %
Table        Colonne                      
products     Color             12445 69.37
transactions Color           4350783 67.80
customers    Job Title        584185 35.55
products     Sizes              2070 11.54
transactions Size             413102  6.44
discounts    Sub Category         10  5.52
             Category             10  5.52

Quatre cas à investiguer :

**`Color` / `Size` dans `transactions`** — peut-on les récupérer depuis `products` via `Product ID` ?

In [8]:
tx_with_product = transactions[["Product ID", "Color", "Size"]].merge(
    products[["Product ID", "Color", "Sizes"]],
    on="Product ID", how="left", suffixes=("", "_product"),
)
print("Color récupérable depuis products :",
      (tx_with_product["Color"].isna() & tx_with_product["Color_product"].notna()).sum())
print("Size récupérable depuis products  :",
      (tx_with_product["Size"].isna() & tx_with_product["Sizes"].notna()).sum())

Color récupérable depuis products : 0
Size récupérable depuis products  : 0


In [9]:
# Sizes manquant dans products : quelles sous-catégories ?
products.loc[products["Sizes"].isna(), "Sub Category"].value_counts()

Sub Category
Accessories    2070
Name: count, dtype: int64

In [10]:
# Job Title : existe-t-il une modalité dominante qui permettrait une imputation ?
job_share = customers["Job Title"].value_counts(normalize=True)
print(f"{customers['Job Title'].nunique()} métiers distincts ; le plus fréquent ne représente que {job_share.iloc[0]:.2%} des valeurs renseignées")

639 métiers distincts ; le plus fréquent ne représente que 0.17% des valeurs renseignées


In [11]:
# Discounts : quelles promotions n'ont pas de catégorie ?
discounts.loc[discounts["Category"].isna(), ["Start", "End", "Discont", "Description"]]

,Start,End,Discont,Description
32,2020-11-27,2020-11-27,0.60,60% discount during our Black Friday Mega Sale
33,2020-12-20,2020-12-31,0.50,50% discount during our Holiday Season Sale
66,2021-11-27,2021-11-27,0.60,60% discount during our Black Friday Mega Sale
67,2021-12-20,2021-12-31,0.50,50% discount during our Holiday Season Sale
100,2022-11-27,2022-11-27,0.60,60% discount during our Black Friday Mega Sale
101,2022-12-20,2022-12-31,0.50,50% discount during our Holiday Season Sale
134,2023-11-27,2023-11-27,0.60,60% discount during our Black Friday Mega Sale
135,2023-12-20,2023-12-31,0.50,50% discount during our Holiday Season Sale
168,2024-11-27,2024-11-27,0.60,60% discount during our Black Friday Mega Sale
169,2024-12-20,2024-12-31,0.50,50% discount during our Holiday Season Sale


**Conclusions :**

| Colonne | Manquants | Diagnostic | Décision |
|---|---|---|---|
| `transactions.Color` | 67,8 % | non récupérable depuis `products` ; taux identique ventes/retours -> absence structurelle, pas un biais | `Unknown` |
| `transactions.Size` | 6,4 % | non récupérable | `Unknown` |
| `products.Sizes` | 11,5 % | uniquement les **accessoires** (taille unique) | `N/A` |
| `products.Color` | 69,4 % | non récupérable | `Unknown` |
| `customers.Job Title` | 35,6 % | des centaines de métiers quasi équiprobables : aucune imputation crédible | `Unknown` (on garde les clients) |
| `discounts.Category` | 10 lignes | Black Friday et soldes de fin d'année = promotions **globales** | `All` |

Aucune suppression de ligne : ces colonnes sont descriptives et ne servent à aucun calcul de montant. En revanche, **`Color` ne sera pas utilisable comme axe d'analyse** (2/3 d'inconnus).

## 3. Doublons

In [12]:
exact_duplicates = transactions[transactions.duplicated(keep=False)]
print("Doublons exacts (lignes en trop) :", transactions.duplicated().sum())
exact_duplicates["Transaction Type"].value_counts()

Doublons exacts (lignes en trop) : 798


Transaction Type
Return    1594
Name: count, dtype: int64

In [13]:
# Après suppression des doublons exacts, la clé naturelle (Invoice ID, Line) est-elle unique ?
deduplicated = transactions.drop_duplicates()
key_collisions = deduplicated[deduplicated.duplicated(subset=["Invoice ID", "Line"], keep=False)]
groups = key_collisions.groupby(["Invoice ID", "Line"])

print(f"Lignes partageant une même clé (Invoice ID, Line) : {len(key_collisions):,}")
print("Types de transaction :", key_collisions["Transaction Type"].value_counts().to_dict())
pd.Series({
    "Même client": (groups["Customer ID"].nunique() == 1).mean(),
    "Même magasin": (groups["Store ID"].nunique() == 1).mean(),
    "Même date": (groups["Date"].nunique() == 1).mean(),
    "Même produit": (groups["Product ID"].nunique() == 1).mean(),
}, name="Part des clés concernées").map("{:.0%}".format)

Lignes partageant une même clé (Invoice ID, Line) : 41,158
Types de transaction : {'Return': 41158}


Même client     100%
Même magasin    100%
Même date        12%
Même produit     35%
Name: Part des clés concernées, dtype: str

In [14]:
for name, key in [("customers", "Customer ID"), ("products", "Product ID"),
                  ("stores", "Store ID"), ("employees", "Employee ID")]:
    print(f"{name:<10} {key:<12} unique : {raw[name][key].is_unique}")

customers  Customer ID  unique : True
products   Product ID   unique : True
stores     Store ID     unique : True
employees  Employee ID  unique : True


- **798 lignes strictement identiques**, toutes des **retours** : un même retour enregistré 2 ou 3 fois, vraisemblablement un double envoi du système de caisse. Elles sont supprimées (sinon les retours seraient surévalués).
- Les clés primaires des tables de référence sont uniques.
- Après cette suppression, ~41 000 lignes partagent encore une même clé `(Invoice ID, Line)`. Ce sont **toutes des retours** d'un même client dans un même magasin, mais le plus souvent **à des dates différentes** et sur des produits différents : le système réutilise un numéro de retour pour des retours distincts. Ce ne sont pas des doublons, ils sont **conservés** ; la table de faits SQL utilise donc une clé technique `Sale_Key` plutôt que `(Invoice ID, Line)` comme clé primaire.

## 4. Plausibilité des valeurs

In [15]:
transactions[["Unit Price", "Quantity", "Discount", "Line Total"]].describe().T

,count,mean,std,min,25%,50%,75%,max
Unit Price,"6,416,827.00",132.46,185.10,2.00,32.50,51.00,116.50,"1,153.50"
Quantity,"6,416,827.00",1.10,0.40,1.00,1.00,1.00,1.00,3.00
Discount,"6,416,827.00",0.12,0.20,0.00,0.00,0.00,0.25,0.60
Line Total,"6,416,827.00",114.19,211.59,"-3,348.00",24.75,43.50,109.00,"3,460.50"


In [16]:
pd.crosstab(transactions["Transaction Type"], transactions["Line Total"] < 0,
            colnames=["Line Total < 0"])

Line Total < 0,False,True
Transaction Type,,
Return,0,339627
Sale,6077200,0


In [17]:
print("Naissances :", customers["Date Of Birth"].min().date(), "->", customers["Date Of Birth"].max().date())
print("Genre      :", customers["Gender"].value_counts().to_dict())
print("Promotions dont End < Start :", (discounts["End"] < discounts["Start"]).sum())
print("Production Cost <= 0 :", (products["Production Cost"] <= 0).sum())

Naissances : 1949-03-20 -> 2007-03-18
Genre      : {'M': 964562, 'F': 677041, 'D': 1703}
Promotions dont End < Start : 0
Production Cost <= 0 : 0


In [18]:
stores["Country"].unique()

<ArrowStringArray>
[ 'United States',             '中国',    'Deutschland', 'United Kingdom',
         'France',         'España',       'Portugal']
Length: 7, dtype: str

- Aucune quantité, aucun prix ni coût négatif ou nul ; remises entre 0 et 60 %.
- Le signe de `Line Total` est parfaitement cohérent avec le type : **toutes les ventes sont positives, tous les retours négatifs**.
- `Unit Price` va de 2 à 1 153 : l'écart vient des devises (les prix en CNY sont ~7x plus élevés), pas d'erreurs de saisie.
- `Gender = D` (1 703 clients, ~0,1 %) : modalité « divers », conservée telle quelle.
- Les pays (`中国`, `Deutschland`, `España`) et les villes chinoises sont saisis en langue locale -> **traduits en anglais** pour des libellés homogènes dans les analyses et le dashboard.

## 5. Cohérence financière

Règle attendue : `Line Total = Unit Price x Quantity x (1 - Discount)` (avec un signe négatif pour les retours).

In [19]:
sign = transactions["Transaction Type"].map({"Sale": 1, "Return": -1})
expected = sign * transactions["Unit Price"] * transactions["Quantity"] * (1 - transactions["Discount"])
inconsistent = (transactions["Line Total"] - expected).abs() > 0.05

inconsistent.groupby(transactions["Transaction Type"]).agg(["sum", "mean"]).rename(
    columns={"sum": "Lignes incohérentes", "mean": "Part"}
)

,Lignes incohérentes,Part
Transaction Type,,
Return,98295,0.29
Sale,0,0.00


Les ventes sont toutes cohérentes. Pour les retours, regardons la remise stockée et le ratio remboursé :

In [20]:
returns = deduplicated[deduplicated["Transaction Type"] == "Return"]
sales = deduplicated[deduplicated["Transaction Type"] == "Sale"]

refund_ratio = returns["Line Total"].abs() / (returns["Unit Price"] * returns["Quantity"])

print("Remise stockée sur les retours :", returns["Discount"].unique())
pd.DataFrame({
    "Retours : 1 - ratio remboursé": (1 - refund_ratio).round(2).value_counts(normalize=True),
    "Ventes : remise appliquée": sales["Discount"].value_counts(normalize=True),
}).sort_index()

Remise stockée sur les retours : [0.]


,Retours : 1 - ratio remboursé,Ventes : remise appliquée
0.00,0.71,0.71
0.20,0.02,0.02
0.25,0.02,0.02
0.35,0.04,0.04
0.40,0.02,0.02
0.45,0.04,0.04
0.50,0.14,0.15
0.60,0.01,0.01


**Interprétation :** la remise est **toujours 0 sur les retours**, alors que le montant remboursé correspond exactement au **prix réellement payé** lors de la vente d'origine. `1 - ratio remboursé` ne prend que les valeurs de remise existantes (0 ; 0,20 ; 0,25 ; … ; 0,60), avec une distribution très proche de celle des ventes.

Ce n'est donc pas une anomalie de montant mais une **perte d'information sur la remise** dans le système source. Décision : pour les retours, `Discount` est reconstituée par `1 - |Line Total| / (Unit Price x Quantity)` (arrondie à 2 décimales). La règle de cohérence devient vraie sur 100 % des lignes, sans ajouter de colonne.

## 6. Intégrité référentielle

In [21]:
foreign_keys = {
    "Customer ID": customers, "Product ID": products,
    "Store ID": stores, "Employee ID": employees,
}
pd.Series({
    key: (~transactions[key].isin(reference[key])).sum()
    for key, reference in foreign_keys.items()
}, name="Lignes orphelines")

Customer ID    0
Product ID     0
Store ID       0
Employee ID    0
Name: Lignes orphelines, dtype: int64

In [22]:
# Clients sans aucune transaction
has_no_transaction = ~customers["Customer ID"].isin(transactions["Customer ID"])
print(f"Clients sans transaction : {has_no_transaction.sum():,} ({has_no_transaction.mean():.1%})")

Clients sans transaction : 359,599 (21.9%)


Aucune transaction orpheline : le modèle en étoile pourra imposer des clés étrangères sans perte de lignes.
Une partie des clients n'a **aucune transaction** : ils seront exclus de la segmentation RFM et du churn (qui ne concernent que les acheteurs).

## 7. Synthèse : problèmes -> décisions

| # | Problème | Volume | Décision (appliquée dans `src/etl.py`) |
|---|---|---|---|
| 1 | Doublons exacts (retours enregistrés 2-3 fois) | 798 lignes | Suppression |
| 1b | Numéros de retour réutilisés pour des retours distincts | ~41 000 lignes | Conservés ; clé technique `Sale_Key` en base |
| 2 | Remise non renseignée sur les retours | ~98 000 retours | Reconstitution à partir du montant remboursé |
| 3 | `Color` / `Size` manquants | 68 % / 6 % | `Unknown` (colonnes descriptives) |
| 4 | `Sizes` manquant (accessoires) | 2 070 produits | `N/A` |
| 5 | `Job Title` manquant | 35,6 % | `Unknown` |
| 6 | Promotions globales sans catégorie | 10 lignes | `All` |
| 7 | Pays / villes en langue locale | 3 pays | Traduction en anglais |
| 8 | Montants en 4 devises | 100 % | Conversion USD dans la vue SQL `vw_Fact_Sales_USD` |
| 9 | Mars 2025 incomplet | 18 jours | Exclu des analyses mensuelles et des prévisions |
| 10 | Colonne mal orthographiée (`Discont`) | — | Renommée `Discount` |

**Limites à garder en tête :** données **synthétiques** (emails `@fake_*.com`, comportements très réguliers) : les conclusions valident une méthode plus qu'elles ne décrivent un vrai marché. Les taux de change ne sont pas fournis par la source : la conversion USD repose sur des taux fixes (hypothèse documentée dans `sql/02_create_tables.sql`).